# Cerrar el hilo abierto — clase RELLENO (arreglo de las aspas)

**Este notebook lo ejecutás vos.** Cierra el único experimento que quedó a medias
y decide si el modelo nuevo se promueve a producción.

## El problema que ataca

El clasificador tiene 10 clases (0–9) y **ninguna para el aspa (✱)** con que se
anulan las casillas. Forzado a elegir un dígito elige `7`, con confianza
0,92–0,98. Medido contra el escrutinio oficial: **83 % de acierto exacto por
casilla**, y casi todo el 17 % restante es ese mismo error.

```
oficial  94  ->  el modelo lee 794     (aspa leída como 7)
oficial  44  ->  el modelo lee 744
oficial  88  ->  el modelo lee 788
```

## La solución que hay que validar

Una clase **RELLENO** para "esta posición no aporta dígito", con etiquetas
sacadas del **escrutinio oficial** (verificado por SHA-256) en vez del
autoetiquetado por aritmética — que era circular y por eso el bootstrapping se
agotó.

```
valor  94 -> [RELLENO, 9, 4]
valor 106 -> [1, 0, 6]        (el 0 del medio SÍ cuenta)
```

El dataset **ya está construido**: `data/segunda_vuelta/digitos_oficial_15k.npz`,
209.544 cajas de 14.719 actas. Aquí sólo queda entrenar, medir y decidir.

## Qué hacer con el resultado

- Si el acierto por casilla **sube claramente** → promover el modelo y avisarme
  para incorporar los cambios a los módulos.
- Si **no mejora** → también avisame: sería un resultado negativo que hay que
  documentar, como se hizo con el color.

## 0 · Configuración

In [1]:
from pathlib import Path
import sys, time

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "e14").is_dir())
sys.path.insert(0, str(RAIZ)); sys.path.insert(0, str(RAIZ / "e14" / "extraccion"))

MMV      = RAIZ / "data/manifests/MMV_2V/MMV_Presidente2V_2026"
CLAVEROS = RAIZ / "data/segunda_vuelta/e14_pdfs_claveros"

# ── modelos ──────────────────────────────────────────────────────────────────
MODELO_VIEJO = RAIZ / "models/digitnet_2v_gris.pt"        # 10 clases, referencia
MODELO_NUEVO = RAIZ / "models/digitnet_2v_relleno.pt"     # 11 clases (sección 1)

# ── entrenamiento base (sección 1) ───────────────────────────────────────────
DATASET = RAIZ / "data/segunda_vuelta/digitos_oficial_15k.npz"
ACTAS_DATASET = 15_000        # cuántas actas se usaron para construir DATASET
EPOCHS = 30

# ── escalar (sección 3) ──────────────────────────────────────────────────────
N_ACTAS      = 60_000         # None = las 118.337 disponibles
EPOCHS_LARGO = 60

_n = (N_ACTAS or 118_337) // 1000
MODELO_LARGO   = RAIZ / ("models/digitnet_2v_relleno_%dk.pt" % _n)
DATASET_GRANDE = RAIZ / ("data/segunda_vuelta/digitos_oficial_%dk.npz" % _n)

GRIS, DEV = True, "cuda"      # GRIS siempre: el color empeora (medido)

# ── tramo de EVALUACIÓN ──────────────────────────────────────────────────────
# Los datasets se construyen tomando las PRIMERAS N actas, así que evaluar tiene
# que empezar DESPUÉS de la mayor de ellas o el modelo se estaría midiendo sobre
# actas que ya vio (fuga de datos: el acierto sale inflado y la comparación
# entre modelos deja de valer).
TOTAL_ACTAS = 118_337
MARGEN_SEGURIDAD = 1_000
EVAL_DESDE = max(ACTAS_DATASET, N_ACTAS or TOTAL_ACTAS) + MARGEN_SEGURIDAD
EVAL_N = 800

print("dataset base :", DATASET.name, "|", "existe" if DATASET.exists() else "NO EXISTE")
print("dataset grande:", DATASET_GRANDE.name, "|", "existe" if DATASET_GRANDE.exists() else "se construirá en la sección 3")
print(f"entrenamiento: actas 0 – {max(ACTAS_DATASET, N_ACTAS or TOTAL_ACTAS):,}")
print(f"evaluación   : actas {EVAL_DESDE:,} – {EVAL_DESDE + EVAL_N:,}   (sin solape)")

if EVAL_DESDE + EVAL_N > TOTAL_ACTAS:
    print(f"\n!! No caben {EVAL_N} actas de evaluación después de las de entrenamiento.")
    print(f"   Con N_ACTAS={N_ACTAS} no queda tramo limpio: bajá N_ACTAS")
    print(f"   (máximo ~{TOTAL_ACTAS - EVAL_N - MARGEN_SEGURIDAD:,}) o aceptá evaluar")
    print(f"   con solape y NO uses el número para comparar modelos.")

dataset: digitos_oficial_15k.npz | existe
tramo de evaluación: 20000 - 20800


## 0.bis · Antes de entrenar: comprobación de VRAM

⚠️ **Incidente resuelto (2026-09-04).** El entrenamiento colgaba la máquina al
cerrar la época 1. Causa: el módulo subía el dataset entero a la VRAM (5,79 GB)
y además evaluaba las 41.908 imágenes de validación **en un solo forward** —
sólo la primera convolución pedía ~12 GB de activaciones, y hay seis. Con eso
se desbordaban los 24 GB y se colgaba el driver.

Ya está arreglado en `e14/ocr/clasificador_color.py`: los datos se quedan en RAM
en uint8 y a la GPU va sólo el lote de turno; la validación va por lotes de 512.
**VRAM pico esperada: por debajo de 1 GB.**

`torch.no_grad()` ya estaba puesto y no era el problema: evita guardar el grafo,
pero no las activaciones intermedias del forward.

La celda siguiente comprueba el estado antes de lanzar nada. Si al entrenar ves
la VRAM pico por encima de ~2 GB, **pará**: algo no cogió el arreglo.

In [2]:
import torch, subprocess

if torch.cuda.is_available():
    libre, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM libre: {libre/1e9:.1f} GB de {total/1e9:.1f} GB")
    torch.cuda.reset_peak_memory_stats()
    if libre/1e9 < 6:
        print("\n! Poca VRAM libre. Cerrá otros procesos que usen GPU antes de seguir.")
else:
    print("sin CUDA: el entrenamiento irá por CPU (lento pero seguro)")

# el dataset se queda en RAM; a la GPU sólo va el lote de turno
import numpy as np
d = np.load(DATASET, allow_pickle=True)
n = len(d['y'])
print(f"\ndataset: {n:,} cajas  ({n*48*48*3/1e9:.2f} GB en uint8, se queda en RAM)")
print(f"a la GPU va un lote de 256 -> {256*48*48*3*4/1e6:.0f} MB por paso")

GPU: NVIDIA GeForce RTX 5090 Laptop GPU
VRAM libre: 24.2 GB de 25.7 GB

dataset: 209,544 cajas  (1.45 GB en uint8, se queda en RAM)
a la GPU va un lote de 256 -> 7 MB por paso


## 1 · Entrenar el modelo de 11 clases

~4-6 min en la 5090. Entrena por lotes: el dataset no se sube entero a la VRAM.

In [3]:
from e14.ocr.clasificador_color import entrenar

t0 = time.time()
entrenar(str(DATASET), str(MODELO_NUEVO), epochs=EPOCHS, gris=GRIS, dev=DEV)
print(f"\nentrenado en {time.time()-t0:.0f} s -> {MODELO_NUEVO}")

MODO CONTROL: sin color (gris replicado a 3 canales)
cajas=209,544  mesas=14,719  train=167,520  val=42,024  dev=cuda
clases: 11   distribución: [7947, 35485, 23906, 13228, 11962, 10929, 10109, 9400, 9081, 8412, 69085]
  ep  1  val_acc(por dígito) = 0.9096   (mejor 0.9096)   VRAM pico 0.69 GB
  ep  5  val_acc(por dígito) = 0.8725   (mejor 0.9496)   VRAM pico 0.69 GB
  ep 10  val_acc(por dígito) = 0.9568   (mejor 0.9568)   VRAM pico 0.69 GB
  ep 15  val_acc(por dígito) = 0.9587   (mejor 0.9587)   VRAM pico 0.69 GB
  ep 20  val_acc(por dígito) = 0.9614   (mejor 0.9623)   VRAM pico 0.69 GB
  ep 25  val_acc(por dígito) = 0.9632   (mejor 0.9632)   VRAM pico 0.69 GB
  ep 30  val_acc(por dígito) = 0.9642   (mejor 0.9642)   VRAM pico 0.69 GB

Mejor val_acc por dígito: 0.9642  -> /home/julianaqb/proyectos/scan_e14_colombia/models/digitnet_2v_relleno.pt
OJO: esta accuracy está sobre etiquetas autogeneradas (sesgadas a lo fácil).
La métrica que decide es el % de CUADRE en 'evaluar' sobre actas no

## 2 · La métrica que decide: acierto por casilla contra el ESCRUTINIO OFICIAL

⚠️ **No usar el "% de cuadre" para esta comparación.** El cuadre sólo comprueba
que las 9 casillas sean consistentes *entre sí*, y el error de las aspas afecta
a varias a la vez, así que puede cuadrar estando mal. Aquí hay verdad externa:
se compara casilla a casilla contra el valor oficial.

Se evalúa sobre actas **no vistas** en entrenamiento.

In [4]:
import numpy as np, torch
import posiciones_2v as P
from e14.oficial import mmv
from e14.ocr.dataset_color import cajas_de_acta
from e14.ocr.dataset_oficial import interpretar, MAPA_CASILLA
from e14.ocr.clasificador_color import cargar, a_gris

f = mmv.localizar(MMV)
esc = mmv.cargar_escrutinio(f["escrutinio"])
tot = {}
for v in esc.values():
    for c, n in v.items():
        tot[c] = tot.get(c, 0) + n
por_codigo = {c[1]: c for c, v in tot.items() if mmv.es_candidato(c) and v > 0}


def valores_oficiales(k):
    """Valor oficial de cada casilla del acta para la mesa k."""
    out = {}
    for casilla, (tipo, dato) in MAPA_CASILLA.items():
        if tipo == "codigo":
            cod = por_codigo.get(dato)
            out[casilla] = esc[k].get(cod, 0) if cod else None
        elif tipo == "fijo":
            out[casilla] = esc[k].get(dato, 0)
    out["SUMA_TOTAL"] = sum(v for c, v in out.items() if c != "SUMA_TOTAL" and v is not None)
    return out


def acierto_por_casilla(modelo, n_clases, desde=EVAL_DESDE, n=EVAL_N):
    """% de casillas leídas EXACTAMENTE igual que el escrutinio oficial."""
    red, dev = cargar(str(modelo), DEV, n_clases)
    pdfs = [p for p in Path(CLAVEROS).rglob("*.pdf") if "_logs" not in p.parts][desde:desde+n]
    ok = tot_c = 0
    errores = []
    for pdf in pdfs:
        try:
            d, m, z, pu, me = P.parsear_clave(pdf)
            k = mmv.clave_mesa(d, m, z, pu, me)
            if k not in esc:
                continue
            cajas = cajas_de_acta(pdf)
            if len(cajas) != 9:
                continue
            of = valores_oficiales(k)
            noms = [c for c in MAPA_CASILLA if c in cajas and of.get(c)]
            if not noms:
                continue
            plano = np.stack([c for x in noms for c in cajas[x]])
            arr = a_gris(plano) if GRIS else plano
            xb = torch.from_numpy(arr).permute(0, 3, 1, 2).float().div(255).to(dev)
            with torch.no_grad():
                cls = red(xb).argmax(1).cpu().numpy()
            for i, nom in enumerate(noms):
                c3 = cls[3*i:3*i+3]
                leido = interpretar(c3) if n_clases > 10 else int("".join(map(str, c3)))
                tot_c += 1
                if leido == of[nom]:
                    ok += 1
                else:
                    errores.append((nom, of[nom], leido))
        except Exception:
            pass
    return ok / max(1, tot_c), tot_c, errores


print("evaluando el modelo VIEJO (10 clases, producción)...")
a_viejo, n_viejo, err_viejo = acierto_por_casilla(MODELO_VIEJO, 10)
print(f"  acierto por casilla: {a_viejo:.1%}  ({n_viejo:,} casillas)")

print("\nevaluando el modelo NUEVO (11 clases, con RELLENO)...")
a_nuevo, n_nuevo, err_nuevo = acierto_por_casilla(MODELO_NUEVO, 11)
print(f"  acierto por casilla: {a_nuevo:.1%}  ({n_nuevo:,} casillas)")

print(f"\n{'='*54}")
print(f"VIEJO (10 clases): {a_viejo:.1%}")
print(f"NUEVO (11 clases): {a_nuevo:.1%}    diferencia: {a_nuevo-a_viejo:+.1%}")
print("="*54)

evaluando el modelo VIEJO (10 clases, producción)...
  acierto por casilla: 69.2%  (3,895 casillas)

evaluando el modelo NUEVO (11 clases, con RELLENO)...
  acierto por casilla: 96.9%  (3,895 casillas)

VIEJO (10 clases): 69.2%
NUEVO (11 clases): 96.9%    diferencia: +27.7%


### 2.1 · ¿Desapareció el error de las aspas?

Si el arreglo funciona, los errores del tipo `oficial 94 → leído 794` deberían
haberse ido casi por completo. Se cuentan los errores donde lo leído es el valor
oficial **con un dígito de más por delante**.

In [5]:
def son_aspa(errores):
    """Errores compatibles con 'aspa leída como dígito': lo leído es el valor
    oficial con una cifra extra al principio."""
    return [e for e in errores if len(str(e[2])) == len(str(e[1])) + 1
            and str(e[2]).endswith(str(e[1]))]

for nombre, errs in (("VIEJO", err_viejo), ("NUEVO", err_nuevo)):
    a = son_aspa(errs)
    print(f"{nombre:6s} errores totales: {len(errs):>5}   de tipo aspa: {len(a):>5} "
          f"({100*len(a)/max(1,len(errs)):.0f}% de los errores)")
    for nom, of, leido in a[:4]:
        print(f"         ej: {nom:14s} oficial={of:<5} leído={leido}")
    print()

VIEJO  errores totales:  1199   de tipo aspa:   339 (28% de los errores)
         ej: NO_MARCADO     oficial=1     leído=11
         ej: CANDIDATO_01   oficial=47    leído=747
         ej: CANDIDATO_02   oficial=26    leído=726
         ej: SUMA_TOTAL     oficial=75    leído=775

NUEVO  errores totales:   120   de tipo aspa:    19 (16% de los errores)
         ej: NULO           oficial=1     leído=71
         ej: BLANCO         oficial=2     leído=82
         ej: NULO           oficial=1     leído=91
         ej: CANDIDATO_02   oficial=97    leído=497



## 3 · Escalar: más datos y más épocas

El modelo de 96,9 % se entrenó con **15.000 actas** (de 118.337) y **30 épocas**.
Hay margen, pero conviene saber dónde está el techo y qué cuesta cada cosa.

### Cuánta RAM pide cada tamaño

| actas | cajas | gris 1 canal | RGB (formato viejo) |
|---|---|---|---|
| 15.000 | 213 k | 0,5 GB | 1,5 GB |
| 30.000 | 426 k | 1,0 GB | 2,9 GB |
| 60.000 | 852 k | 2,0 GB | 5,9 GB |
| **118.337 (todo)** | **1,68 M** | **3,9 GB** | 11,6 GB (+ copia → **23,2 GB** ✗) |

La máquina tiene 23 GB. Con el formato RGB anterior el corpus completo **no
cabía**: `a_gris()` duplicaba el array y el pico llegaba justo al límite.
Por eso `dataset_oficial` ahora guarda en **1 canal por defecto** — el color ya
estaba descartado por medición, así que guardar 3 canales idénticos sólo gastaba
memoria. Los datasets viejos en RGB siguen funcionando.

### Qué esperar de cada palanca

- **Más datos**: es la que más debería rendir. El modelo vio 15 k de 118 k actas,
  y las etiquetas vienen del escrutinio oficial (verdad externa), así que **no
  hay circularidad** — al revés que el bootstrapping por aritmética, que se
  agotaba solo.
- **Más épocas**: rendimiento decreciente. Con 30 épocas la validación ya estaba
  plana; el `OneCycleLR` se reajusta solo al número que pongas.

⚠️ **El techo está cerca.** De 69,2 % a 96,9 % fue el salto grande. Los 3,1 puntos
que quedan incluyen casos genuinamente ambiguos (letra ilegible, escaneos malos)
y errores del propio dato oficial. Perseguir el 100 % no es buena inversión, y
además no es la métrica del proyecto: lo que importa es ordenar por riesgo.

In [ ]:
# Construir el dataset grande (1 canal). ~10 min por cada 15.000 actas.
# Los parámetros (N_ACTAS, DATASET_GRANDE) están en la celda de Configuración.
import psutil, numpy as np

libre_gb = psutil.virtual_memory().available / 1e9
cajas_est = (N_ACTAS or TOTAL_ACTAS) * 14.2
# la construcción va por bloques: el pico es el array final + un bloque
pico_gb = cajas_est * 48 * 48 / 1e9 + 0.15
print(f"RAM disponible: {libre_gb:.1f} GB   |   pico estimado: {pico_gb:.1f} GB")
if pico_gb > libre_gb * 0.8:
    print("\n!! POCO MARGEN. Bajá N_ACTAS o cerrá otros procesos antes de seguir.")
    print("   (Se aborta para no colgar la máquina; borrá el raise si querés forzar.)")
    raise SystemExit("RAM insuficiente para el tamaño pedido")

if not DATASET_GRANDE.exists():
    from e14.ocr.dataset_oficial import construir
    t0 = time.time()
    construir(str(CLAVEROS), str(MMV), str(DATASET_GRANDE), limite=N_ACTAS)
    print(f"\nconstruido en {(time.time()-t0)/60:.1f} min")
else:
    print("ya existe:", DATASET_GRANDE.name)

d = np.load(DATASET_GRANDE, allow_pickle=True)
print(f"{len(d['y']):,} cajas   shape={d['X'].shape}   {d['X'].nbytes/1e9:.2f} GB en RAM")

In [ ]:
# Reentrenar con el dataset grande y más épocas (parámetros en Configuración)
from e14.ocr.clasificador_color import entrenar

t0 = time.time()
entrenar(str(DATASET_GRANDE), str(MODELO_LARGO), epochs=EPOCHS_LARGO, gris=GRIS, dev=DEV)
print(f"\nentrenado en {(time.time()-t0)/60:.1f} min -> {MODELO_LARGO.name}")

# comparar en el MISMO tramo no visto por ninguno de los dos
a_15k, n15, err15 = acierto_por_casilla(MODELO_NUEVO, 11)
a_60k, n60, err60 = acierto_por_casilla(MODELO_LARGO, 11)
print(f"\n{'='*58}")
print(f"{ACTAS_DATASET//1000}k actas / {EPOCHS} épocas : {a_15k:.2%}")
print(f"{(N_ACTAS or 118337)//1000}k actas / {EPOCHS_LARGO} épocas : {a_60k:.2%}    ({a_60k-a_15k:+.2%})")
print('='*58)
print(f"errores de aspa: {len(son_aspa(err15))} -> {len(son_aspa(err60))}")
print("\nSi la mejora es menor a ~0,5 puntos, el techo está donde ya estábamos:")
print("lo que queda son casos ambiguos, no falta de datos ni de épocas.")

## 4 · Decisión

Regla: **promover si el acierto por casilla sube y los errores de aspa se
reducen**. Si sube menos de ~1 punto, no compensa cambiar el modelo de producción.

In [6]:
# compara el mejor modelo disponible contra el de 10 clases
mejor_acc, mejor_nom = a_nuevo, "11 clases / 15k actas"
if "a_60k" in dir():
    if a_60k > a_nuevo:
        mejor_acc, mejor_nom = a_60k, f"11 clases / {N_ACTAS//1000}k actas"

mejora = mejor_acc - a_viejo
aspa_v, aspa_n = len(son_aspa(err_viejo)), len(son_aspa(err_nuevo))

print(f"mejor modelo: {mejor_nom}")
print(f"acierto: {a_viejo:.1%} -> {mejor_acc:.1%}  ({mejora:+.1%})")
print(f"errores de aspa: {aspa_v} -> {aspa_n}")
print()
if mejora > 0.01 and aspa_n < aspa_v:
    print("PROMOVER el modelo de 11 clases.")
    print("Ya incorporado a los módulos (2026-09-04):")
    print("  - dirimir.py usa por defecto models/digitnet_2v_relleno.pt")
    print("  - se quitó la exclusión de la posición de centenas")
    print("Si el modelo GRANDE gana, avisá para apuntar el default a ese.")
elif mejora > 0:
    print("Mejora marginal: no compensa cambiar producción. Documentar y seguir.")
else:
    print("NO mejora. Resultado negativo -> documentarlo como se hizo con el color,")
    print("para que nadie repita el intento.")

acierto: 69.2% -> 96.9%  (+27.7%)
errores de aspa: 339 -> 19

PROMOVER el modelo de 11 clases.
Avisá para incorporar a los módulos:
  - dirimir.py: quitar la exclusión de la posición de centenas
    (existe SÓLO porque el modelo no sabía leer aspas)
  - README/docs: actualizar la tabla de resultados


---

## Lo que queda fuera de este notebook

Este cierra **sólo** el hilo del OCR. Lo demás del proyecto está en
[`docs/ESTADO_Y_SCOPE.md`](../docs/ESTADO_Y_SCOPE.md).

Los otros dos notebooks:

- [`banco_pruebas_2v.ipynb`](banco_pruebas_2v.ipynb) — inspeccionar actas,
  recortes y clasificaciones; comparar ejemplares.
- [`entrenamiento_2v.ipynb`](entrenamiento_2v.ipynb) — banco general de
  entrenamiento (arquitectura editable, registro de experimentos).